## 🎯 Learning Objectives
* Design and implement a multi-agent RAG system using LangGraph.
* Integrate various tools (vector store, web search) within a LangGraph workflow.
* Implement self-correction and adaptive routing mechanisms in an agentic RAG system.
* Set up and utilize RAGAS for evaluating the performance of an agentic RAG system.
* Understand the practical considerations for building and evaluating production-ready agentic RAG.


## ADV04-L12: Exercise: Build a Full Agentic RAG System with Evals

**Track**: Agentic AI & Automation Tools
**Course**: ADV-04 — Building Agentic RAG Systems with LangGraph
**Section**: Advanced Patterns

### Exercise Task

Your task is to build a sophisticated, agentic Retrieval Augmented Generation (RAG) system using LangGraph. This system should be capable of intelligently answering complex user queries by leveraging multiple tools and incorporating self-correction mechanisms. The final system must also include a robust evaluation pipeline using RAGAS to measure its performance.

**System Requirements:**

1.  **Multi-Tool Integration**: Your agentic system must be able to utilize at least two distinct tools:
    *   **Vector Store Retriever**: For retrieving information from a pre-indexed knowledge base.
    *   **Web Search Tool**: For fetching up-to-date or external information not present in the vector store.
2.  **Agentic Workflow (LangGraph)**: Design a LangGraph workflow that includes at least the following conceptual agents/nodes:
    *   **Query Router/Planner**: An initial agent that analyzes the user's query and decides which tool(s) to use (e.g., vector store, web search, or both) or if a direct answer is possible. It might also break down complex queries.
    *   **Retrieval Agent**: Executes the vector store search based on the router's decision.
    *   **Web Search Agent**: Executes the web search based on the router's decision.
    *   **Answer Synthesis Agent**: Generates a comprehensive answer by synthesizing information from the activated tools.
    *   **Critique/Self-Correction Agent**: Evaluates the generated answer for faithfulness, relevance, and completeness. If the answer is deemed unsatisfactory, it should trigger a re-planning or re-execution step (e.g., try a different tool, refine the query, or perform another search).
3.  **Evaluation Pipeline (RAGAS)**: Implement an evaluation loop that:
    *   Loads a small set of test questions with ground truth answers and contexts (you can create a mock dataset for this).
    *   Runs each test question through your agentic RAG system.
    *   Uses RAGAS to compute metrics such as faithfulness, answer relevance, context relevance, and potentially others.
    *   Prints or logs the evaluation results.

**Technical Considerations (2026 Ready):**

*   Utilize the latest versions of `langchain`, `langgraph`, and `ragas`.
*   Leverage modern LLMs (e.g., OpenAI's latest models or equivalent open-source models via `ollama` or `vllm` if you have local setup).
*   Ensure your code is modular, well-commented, and follows best practices.

### Evaluation Criteria:

*   **Correctness**: Does the LangGraph workflow execute as intended? Are the tools correctly integrated?
*   **Robustness**: Does the system handle various query types (simple, complex, requiring external knowledge)? Does the self-correction mechanism work?
*   **Completeness**: Are all requirements met (multi-tool, agentic workflow, RAGAS evaluation)?
*   **Code Quality**: Is the code clean, readable, well-commented, and modular?
*   **Performance (Evals)**: Are the RAGAS metrics computed correctly? Does the system demonstrate reasonable performance on the mock dataset?

Good luck! This exercise will solidify your understanding of building advanced agentic RAG systems.


In [ ]:
# Install necessary libraries (ensure you have these in your environment)
# !pip install -qU langchain langchain-openai langgraph ragas chromadb beautifulsoup4 requests

import os
from typing import List, Dict, Any, Optional

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, END

# For RAGAS evaluation
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevance, context_relevance, context_recall

# --- Environment Setup ---
# Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Ensure API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not set. Please set it.")

# --- Initialize LLMs and Embeddings ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# --- Mock Knowledge Base Setup ---
# In a real scenario, this would be loaded from a persistent store or external source.
knowledge_base_docs = [
    "The capital of France is Paris. Paris is known for the Eiffel Tower and the Louvre Museum.",
    "The Amazon rainforest is the largest tropical rainforest in the world, spanning several South American countries, primarily Brazil.",
    "Quantum computing uses quantum-mechanical phenomena like superposition and entanglement to perform computations.",
    "The latest advancements in AI in 2026 include highly efficient multimodal models and advanced agentic frameworks like LangGraph.",
    "Agentic RAG systems combine the power of large language models with retrieval mechanisms and autonomous decision-making agents.",
    "LangGraph is a library for building stateful, multi-actor applications with LLMs, inspired by Apache Flink and directed acyclic graphs.",
    "The primary goal of corrective RAG is to identify and fix issues in the retrieval or generation process to improve answer quality.",
    "Self-RAG systems allow the LLM to self-reflect and improve its own retrieval and generation steps iteratively."
]

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunked_docs = text_splitter.create_documents(knowledge_base_docs)

# Create a Chroma vector store
vectorstore = Chroma.from_documents(documents=chunked_docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

# --- Tools Setup ---
web_search_tool = DuckDuckGoSearchRun()

# --- Define LangGraph State ---
# This defines the state of our graph. It will be passed around to each node.
class AgentState(TypedDict):
    question: str
    chat_history: List[BaseMessage]
    generation: Optional[str]
    documents: List[Document]
    web_search_result: Optional[str]
    critique: Optional[str]

# --- Helper Functions (for nodes) ---

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

print("Setup complete: LLM, Embeddings, Vector Store, Retriever, and Web Search Tool initialized.")
print(f"Knowledge base contains {len(chunked_docs)} chunks.")


### Implement Your Agentic RAG System

Now it's your turn to build the agentic RAG system. Use the provided setup code (LLM, embeddings, vector store, tools) and the `AgentState` definition. Your implementation should follow the requirements outlined in the exercise task, including:

1.  **Define Nodes**: Create Python functions for each conceptual agent (Query Router/Planner, Retrieval Agent, Web Search Agent, Answer Synthesis Agent, Critique/Self-Correction Agent). Each function will take the `AgentState` as input and return an updated `AgentState`.
2.  **Define Edges**: Connect your nodes using `StateGraph`'s `add_edge` and `add_conditional_edges` methods.
3.  **Build the Graph**: Compile your nodes and edges into a runnable LangGraph `StateGraph`.
4.  **Implement Evaluation**: Create a small mock dataset for evaluation and use RAGAS to assess your system's performance.

Feel free to experiment with different prompt engineering strategies for each agent to optimize their behavior. Remember to add comments to your code explaining your design choices.


In [ ]:
from typing import TypedDict, List, Dict, Any, Optional
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

# --- Define LangGraph State (re-defined for clarity in solution cell) ---
class AgentState(TypedDict):
    question: str
    chat_history: List[BaseMessage]
    generation: Optional[str]
    documents: List[Document]
    web_search_result: Optional[str]
    critique: Optional[str]
    # Add a field to track the decision of the router
    tool_decision: Optional[str] # 'vector_store', 'web_search', 'both', 'direct_answer'

# --- Nodes for the Agentic RAG System ---

# 1. Query Router/Planner Agent
def route_query(state: AgentState) -> Dict[str, Any]:
    """
    Decides whether to use the vector store, web search, both, or attempt a direct answer.
    """
    print("---ROUTE QUERY---")
    question = state["question"]
    
    # Prompt for the router LLM
    router_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an intelligent routing agent. Based on the user's question, decide whether to perform a vector store search, a web search, both, or if the question can be answered directly without external tools. Respond with 'vector_store', 'web_search', 'both', or 'direct_answer'.\n\nExamples:\nQuestion: 'What is LangGraph?' -> vector_store\nQuestion: 'What are the latest AI advancements in 2026?' -> web_search\nQuestion: 'Who won the last FIFA World Cup?' -> web_search\nQuestion: 'Tell me about the Amazon rainforest.' -> vector_store\nQuestion: 'Compare LangGraph with the latest AI advancements.' -> both\nQuestion: 'What is 2+2?' -> direct_answer"),
        ("human", "{question}")
    ])
    
    # Chain for routing
    router_chain = router_prompt | llm | StrOutputParser()
    
    decision = router_chain.invoke({"question": question})
    decision = decision.strip().lower()
    
    print(f"Router decision: {decision}")
    return {"tool_decision": decision}

# 2. Retrieval Agent (Vector Store)
def retrieve_documents(state: AgentState) -> Dict[str, Any]:
    """
    Retrieves documents from the vector store based on the question.
    """
    print("---RETRIEVE DOCUMENTS---")
    question = state["question"]
    documents = retriever.invoke(question)
    print(f"Retrieved {len(documents)} documents from vector store.")
    return {"documents": documents}

# 3. Web Search Agent
def web_search(state: AgentState) -> Dict[str, Any]:
    """
    Performs a web search based on the question.
    """
    print("---WEB SEARCH---")
    question = state["question"]
    web_result = web_search_tool.invoke({"query": question})
    print(f"Web search performed. Result length: {len(web_result)} characters.")
    return {"web_search_result": web_result}

# 4. Answer Synthesis Agent
def generate_answer(state: AgentState) -> Dict[str, Any]:
    """
    Generates the final answer based on retrieved documents, web search results, and the question.
    """
    print("---GENERATE ANSWER---")
    question = state["question"]
    documents = state["documents"]
    web_search_result = state["web_search_result"]
    
    context = ""
    if documents:
        context += "\n\nRetrieved Documents:\n" + format_docs(documents)
    if web_search_result:
        context += "\n\nWeb Search Results:\n" + web_search_result
    
    # If no context from tools, it might be a direct answer or a failure to retrieve
    if not context.strip():
        context = "No relevant information found from tools. Try to answer based on general knowledge if possible, or state that information is unavailable."

    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are an expert AI assistant. Answer the user's question comprehensively and accurately based *only* on the provided context. If the context does not contain enough information, state that you cannot answer fully based on the provided information. Do not make up answers.\n\nContext:\n{context}"),
        ("human", "{question}")
    ])
    
    answer_chain = prompt_template | llm | StrOutputParser()
    
    generation = answer_chain.invoke({"context": context, "question": question})
    print(f"Generated answer (first 100 chars): {generation[:100]}...")
    return {"generation": generation}

# 5. Critique/Self-Correction Agent
def critique_answer(state: AgentState) -> Dict[str, Any]:
    """
    Critiques the generated answer for faithfulness, relevance, and completeness.
    Decides if re-planning or re-execution is needed.
    """
    print("---CRITIQUE ANSWER---")
    question = state["question"]
    generation = state["generation"]
    documents = state["documents"]
    web_search_result = state["web_search_result"]
    
    context_sources = ""
    if documents:
        context_sources += "\n\nRetrieved Documents:\n" + format_docs(documents)
    if web_search_result:
        context_sources += "\n\nWeb Search Results:\n" + web_search_result

    critique_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert critique agent. Evaluate the generated answer based on the original question and the provided context. Assess its faithfulness to the context, relevance to the question, and completeness. If the answer is satisfactory, respond with 'SATISFACTORY'. If it's not, explain why and suggest a next step (e.g., 'REVISE_ANSWER', 'RETRY_WEB_SEARCH', 'RETRY_VECTOR_STORE', 'REPLAN').\n\nQuestion: {question}\nContext:\n{context_sources}\n\nGenerated Answer: {generation}\n\nCritique:"),
        ("human", "Is the answer satisfactory? If not, why, and what should be the next step?")
    ])
    
    critique_chain = critique_prompt | llm | StrOutputParser()
    
    critique_text = critique_chain.invoke({
        "question": question,
        "context_sources": context_sources,
        "generation": generation
    })
    
    print(f"Critique: {critique_text}")
    return {"critique": critique_text}

# --- Conditional Edges (Decision Logic) ---

def decide_to_search(state: AgentState) -> str:
    """
    Conditional edge for routing based on the router's decision.
    """
    print("---DECIDE TO SEARCH---")
    decision = state["tool_decision"]
    if decision == 'vector_store':
        return "retrieve_documents"
    elif decision == 'web_search':
        return "web_search"
    elif decision == 'both':
        return "retrieve_and_web_search"
    elif decision == 'direct_answer':
        return "generate_answer" # Attempt to answer directly without tools
    else:
        # Fallback or error handling
        print(f"Unexpected router decision: {decision}. Defaulting to web search.")
        return "web_search"

def decide_to_replan(state: AgentState) -> str:
    """
    Conditional edge for self-correction based on the critique.
    """
    print("---DECIDE TO REPLAN---")
    critique = state["critique"]
    if "SATISFACTORY" in critique.upper():
        print("Critique: SATISFACTORY. Ending.")
        return "end"
    elif "RETRY_WEB_SEARCH" in critique.upper():
        print("Critique suggests RETRY_WEB_SEARCH.")
        return "web_search"
    elif "RETRY_VECTOR_STORE" in critique.upper():
        print("Critique suggests RETRY_VECTOR_STORE.")
        return "retrieve_documents"
    elif "REVISE_ANSWER" in critique.upper() or "REPLAN" in critique.upper():
        print("Critique suggests REVISE_ANSWER/REPLAN. Re-generating answer.")
        return "generate_answer" # Re-attempt generation with existing context
    else:
        print("Critique unclear or suggests ending. Ending.")
        return "end"

# --- Build the LangGraph Workflow ---

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("route_query", route_query)
workflow.add_node("retrieve_documents", retrieve_documents)
workflow.add_node("web_search", web_search)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("critique_answer", critique_answer)

# Set entry point
workflow.set_entry_point("route_query")

# Add edges
workflow.add_conditional_edges(
    "route_query",
    decide_to_search,
    {
        "retrieve_documents": "retrieve_documents",
        "web_search": "web_search",
        "both": "retrieve_documents", # Start with retrieval, then web search
        "direct_answer": "generate_answer"
    }
)

# If 'both' was decided, after retrieval, go to web search
workflow.add_edge("retrieve_documents", "web_search")

# After web search (whether direct or after retrieval), go to generate answer
workflow.add_edge("web_search", "generate_answer")

# After generating an answer, always critique it
workflow.add_edge("generate_answer", "critique_answer")

# From critique, decide whether to loop back or end
workflow.add_conditional_edges(
    "critique_answer",
    decide_to_replan,
    {
        "web_search": "web_search",
        "retrieve_documents": "retrieve_documents",
        "generate_answer": "generate_answer",
        "end": END
    }
)

# Compile the graph
app = workflow.compile()

print("LangGraph workflow compiled successfully.")

# --- Test the system with a few queries ---
print("\n--- Testing the Agentic RAG System ---")

# Example 1: Question answerable by vector store
print("\nQuery 1: What is LangGraph and what is its inspiration?")
inputs = {"question": "What is LangGraph and what is its inspiration?", "chat_history": []}
for s in app.stream(inputs):
    print(s)
    print("----")
final_state_1 = app.invoke(inputs)
print(f"Final Answer 1: {final_state_1['generation']}")

# Example 2: Question requiring web search
print("\nQuery 2: Who is the current CEO of Google in 2026?")
inputs = {"question": "Who is the current CEO of Google in 2026?", "chat_history": []}
for s in app.stream(inputs):
    print(s)
    print("----")
final_state_2 = app.invoke(inputs)
print(f"Final Answer 2: {final_state_2['generation']}")

# Example 3: Question requiring both (or complex routing)
print("\nQuery 3: How do agentic RAG systems relate to the latest AI advancements?")
inputs = {"question": "How do agentic RAG systems relate to the latest AI advancements?", "chat_history": []}
for s in app.stream(inputs):
    print(s)
    print("----")
final_state_3 = app.invoke(inputs)
print(f"Final Answer 3: {final_state_3['generation']}")

# --- RAGAS Evaluation Pipeline ---
print("\n--- Starting RAGAS Evaluation ---")

# 1. Create a mock dataset for evaluation
# In a real scenario, this would be a larger, more diverse dataset.
# Ensure ground_truths are accurate and contexts are what the system *should* retrieve.

eval_data = {
    "question": [
        "What is the capital of France?",
        "What is the Amazon rainforest known for?",
        "Explain quantum computing in simple terms.",
        "What are agentic RAG systems?",
        "Who is the current President of the United States in 2026?"
    ],
    "ground_truths": [
        ["Paris is the capital of France, famous for landmarks like the Eiffel Tower."],
        ["The Amazon rainforest is the largest tropical rainforest globally, primarily in Brazil."],
        ["Quantum computing uses quantum mechanics (superposition, entanglement) for computations."],
        ["Agentic RAG systems combine LLMs, retrieval, and autonomous agents for decision-making."],
        ["The current President of the United States in 2026 is Joe Biden."] # Assuming no change from 2024 for mock data
    ]
}

# Convert to RAGAS Dataset format
eval_dataset = Dataset.from_dict(eval_data)

# 2. Define a function to run the RAG system for RAGAS
def run_rag_system(row: Dict[str, Any]) -> Dict[str, Any]:
    question = row["question"]
    inputs = {"question": question, "chat_history": []}
    
    # Invoke the graph to get the final state
    final_state = app.invoke(inputs)
    
    # Extract information needed for RAGAS
    answer = final_state.get("generation", "")
    
    # Combine documents and web search results for context
    retrieved_docs = final_state.get("documents", [])
    web_search_res = final_state.get("web_search_result", "")
    
    contexts = [doc.page_content for doc in retrieved_docs]
    if web_search_res:
        # For web search, we might need to chunk it or summarize if too long
        # For simplicity, we'll add it as a single context string here.
        # In a real scenario, you might process this more carefully.
        contexts.append(web_search_res)
    
    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "ground_truths": row["ground_truths"]
    }

# 3. Apply the RAG system to the evaluation dataset
print("Running RAG system on evaluation dataset...")
rag_results = []
for i, row in enumerate(eval_dataset):
    print(f"Processing eval question {i+1}/{len(eval_dataset)}: {row['question']}")
    rag_results.append(run_rag_system(row))

# Convert results back to RAGAS Dataset format
ragas_dataset = Dataset.from_list(rag_results)

# 4. Define RAGAS metrics
metrics = [
    faithfulness,
    answer_relevance,
    context_relevance,
    context_recall
]

# 5. Evaluate the system
print("Evaluating with RAGAS...")
result = evaluate(
    ragas_dataset,
    metrics=metrics,
    llm=llm, # Use the same LLM for RAGAS evaluations
    embeddings=embeddings
)

# 6. Print evaluation results
print("\n--- RAGAS Evaluation Results ---")
print(result)
print("\nDetailed metrics:")
print(result.to_pandas())

print("\nAgentic RAG system with evaluation pipeline complete!")
